In [5]:
import sys, os, matplotlib.pyplot as plt, pandas as pd, plotly.graph_objects as go


parent_dir = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
sys.path.append(parent_dir)
from tqdm import tqdm
import re, shutil, random

from one_sided_reflected_geometric_brownian_motion.data_load import *



In [6]:
# First : checkout all data


# ---- configure this ----
SOURCES_TXT = Path("ECA_nonblend_rr_all_europe_06_10_25") / "sources.txt"

df_stations = load_all_stations_with_coords(sources_path = SOURCES_TXT)
print(df_stations.head())
print(f"\nTotal stations loaded: {len(df_stations):,}")

### Custom filtering
# Filter out Sicilia data
mask = (df_stations.country == "IT") & (df_stations.lat < 41) & (df_stations.lon < 10)
df_stations = df_stations[~mask]
# Filter out a part of North Italy stations
p = 0.8 # Fraction of stations to filter out
rng = np.random.default_rng(42)   # remove or change seed if you want different draws each run
mask = pd.Series(False, index=df_stations.index)                 # False everywhere
it_idx = (df_stations['country'] == 'IT') & (df_stations.lat > 43.3)  & (df_stations.lat < 47)                  # subset
mask.loc[it_idx] = rng.random(it_idx.sum()) < p 
df_stations = df_stations[~mask]
# Filter out Luxembourg data
mask = (df_stations.country == "LU")
df_stations = df_stations[~mask]

# Optional: save for exploration
out_csv = Path("all_stations_europe_metadata.csv")
df_stations.to_csv(out_csv, index=False)
print(f"Saved: {out_csv}")

    souid               city country        lat        lon
0  100020           KARLSTAD      SE  59.350000  13.466667
1  100023   KARLSTAD-AIRPORT      SE  59.444444  13.337500
2  100026         OESTERSUND      SE  63.183056  14.483056
3  100035  GRAZ-UNIVERSITAET      AT  47.077778  15.448889
4  100037     INNSBRUCK-UNIV      AT  47.260000  11.384167

Total stations loaded: 16,497
Saved: all_stations_europe_metadata.csv


In [7]:
# Filter data 1

import os, re, csv, random, shutil
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd

# -------------------- CONFIG --------------------
METADATA_CSV = Path("all_stations_europe_metadata.csv")  # <-- your metadata CSV path
DATA_DIR = Path("ECA_nonblend_rr_all_europe_06_10_25")  # folder containing RR_SOUID*.txt
OUT_CSV = Path("all_stations_metadata_filtered.csv")

# LIST_COUNTRY_TO_KEEP = {'FR','DE','IT','ES','PT','GR'}
LIM_PER_COUNTRY = 500


# spatial bounds (exclusive, as requested)
LAT_MIN, LAT_MAX = 25.0, 60.0
LON_MIN, LON_MAX = -10.0, 30.0

# Missing threshold (set APPLY_TO_POST=True to enforce only on post-1945)
APPLY_TO_POST = False

# -------------------- LOAD METADATA CSV --------------------
df = pd.read_csv(METADATA_CSV)

bounds = {"lat_min": LAT_MIN, "lat_max": LAT_MAX, "lon_min": LON_MIN, "lon_max": LON_MAX}

cand_df = process_stations(
    df,
    bounds=bounds,
    per_country_limit=LIM_PER_COUNTRY,
    country_whitelist=None,
    city_key_func=default_city_key,      # or a custom func like: lambda s: s.lower().split()[0]
    random_state=42,
    out_csv_path=OUT_CSV
)


cand_df

,souid,city,country,lat,lon
0,267762,TINEO SOUTU,ES,43.290278,-6.386667
1,186597,ALTENSTADT/WALDNAAB,DE,49.714167,12.170833
2,139154,HANGSTAD,SE,59.860000,11.970000
3,185835,LOHBERG,DE,49.183056,13.100000
4,120404,PONTEVEDRA INSTITUTO,ES,42.430556,-8.646944
...,...,...,...,...,...
11342,218439,GT-CUMBRAE-MILLPORT,GB,55.748889,-4.907778
11345,175075,VAIKE-MAARJA,EE,59.141389,26.230556
11353,259344,CSENGER,HU,47.845556,22.663056
11356,202053,HORNI,NO,59.915278,10.460278


In [ ]:
### Keep only datasets with sufficient data after a given date
START_YEAR_EXCLUSIVE = 1945         # "after 1945" -> 1946+
MIN_YEARS_WITH_DATA = 50            # at least 50 distinct years with >=1 valid day
MAX_NAN_FRACTION = 0.05             # â‰¤ 5% NaNs, on the post-1945 subset
YEARS_MODE = "distinct_years"       # "distinct_years" or "days_count"
# If you want the stricter interpretation (>= ~50*365 valid days), set:
# YEARS_MODE = "days_count"
# and MIN_VALID_DAYS = 50*365

MIN_VALID_DAYS = MIN_YEARS_WITH_DATA*365  # used only if YEARS_MODE == "days_count"
MAX_PER_COUNTRY = 500  # density-based filtering will reduce further in the next step

kept, rejected = filter_ecad_rr(
    cand_df,
    src_dir="ECA_nonblend_rr_all_europe_06_10_25",
    dst_dir="ecad_data_europe_filtered_post1945_50y_max5pctnan",
    start_year_exclusive=START_YEAR_EXCLUSIVE,
    min_years_with_data=MIN_YEARS_WITH_DATA,
    max_nan_fraction=MAX_NAN_FRACTION,
    years_mode="distinct_years",  # or "days_count"
    min_valid_days=MIN_VALID_DAYS,
    max_per_country=MAX_PER_COUNTRY,              # NEW: cap on kept rows per country (None -> no cap)
    copy_data = False
)
kept.to_csv("temp_kept_step2.csv")


  0%|          | 0/4655 [00:00<?, ?it/s]

  4%|▍         | 199/4655 [00:40<15:02,  4.94it/s]


KeyboardInterrupt: 

In [8]:
kept = pd.read_csv("temp_kept_step2.csv")
len(kept)

2726

In [17]:
### Step 2: Density-based filtering
# Goal: equalize station density (stations/kmÂ²) across countries by downsampling
# countries with too many stations relative to the median density.

import numpy as np

ISO_TO_COUNTRY = {
    'AL': 'Albania', 'AT': 'Austria', 'BA': 'Bosnia and Herzegovina',
    'BE': 'Belgium', 'BG': 'Bulgaria', 'BY': 'Belarus',
    'CH': 'Switzerland', 'CZ': 'Czech Republic', 'DE': 'Germany',
    'DK': 'Denmark', 'DZ': 'Algeria', 'EE': 'Estonia',
    'ES': 'Spain', 'FI': 'Finland', 'FR': 'France',
    'GB': 'United Kingdom', 'GR': 'Greece', 'HR': 'Croatia',
    'HU': 'Hungary', 'IE': 'Ireland', 'IS': 'Iceland',
    'IT': 'Italy', 'LT': 'Lithuania', 'LU': 'Luxembourg',
    'LV': 'Latvia', 'LY': 'Libya', 'MA': 'Morocco',
    'MD': 'Moldova', 'ME': 'Montenegro', 'MK': 'North Macedonia',
    'NL': 'Netherlands', 'NO': 'Norway', 'PL': 'Poland',
    'PT': 'Portugal', 'RO': 'Romania', 'RS': 'Serbia',
    'SE': 'Sweden', 'SI': 'Slovenia', 'SK': 'Slovakia',
    'TN': 'Tunisia', 'TR': 'Turkey', 'UA': 'Ukraine',
    'XK': 'Kosovo', "RU":"Russia", "MT": "Malta", "EG":"Egypt"
}

# Load country areas; strip comma-separators and trailing footnote digits from names
df_area = pd.read_csv("country_area.csv")
df_area['area_km2'] = df_area['total area square km'].astype(str).str.replace(',', '', regex=False).astype(float)
df_area['country_clean'] = df_area['country'].str.rstrip('0123456789').str.strip()

# Count stations per country and merge with area
counts = kept.groupby('country').size().reset_index(name='count')
counts['country_name'] = counts['country'].map(ISO_TO_COUNTRY)
counts = counts.merge(
    df_area[['country_clean', 'area_km2']].rename(columns={'country_clean': 'country_name'}),
    on='country_name', how='left'
)

counts['density'] = counts['count'] / counts['area_km2']

# median_density = counts['density'].median()
threshold_density = counts['density'].quantile(0.8)
print(f"Median density: {threshold_density:.6f} stations/kmÂ²")
# print(counts[['country', 'country_name', 'count', 'area_km2', 'density']].to_string())

# Target = median_density * area, capped at current count
counts['target'] = (threshold_density * counts['area_km2']).round().astype(int).clip(upper=counts['count'])

# For countries with no area data, keep all stations
counts.loc[counts['area_km2'].isna(), 'target'] = counts.loc[counts['area_km2'].isna(), 'count']

# Subsample each country to the target
kept_density_parts = []
for _, row in counts.iterrows():
    subset = kept[kept['country'] == row['country']]
    target = int(row['target'])
    if len(subset) > target:
        subset = subset.sample(n=target, random_state=42)
    kept_density_parts.append(subset)

kept_density = pd.concat(kept_density_parts).reset_index(drop=True)
kept_density.to_csv("df_candidates_density_filtered.csv", index=False)
print(f"\nBefore density filter: {len(kept)} stations")
print(f"After density filter:  {len(kept_density)} stations")
# print("\nStations per country after filtering:")
# print(kept_density.groupby('country').size().sort_values(ascending=False))


Median density: 0.000596 stations/kmÂ²

Before density filter: 2726 stations
After density filter:  1647 stations


In [20]:
extract_city = ["GOKCEADA","KUSTENDIL", "PALERMO","VANDELLOS"]
for city in extract_city:
    print(city in kept_density.city.tolist())

True
True
True
True


In [28]:
### Step 3: Plot density-filtered stations for manual south-europe selection

kept_south_europe = kept_density[kept_density.lat < 45]
fig = px.scatter_mapbox(
    kept_south_europe,
    lat="lat",
    lon="lon",
    hover_name="city",
    hover_data={"country": True, "years_with_data": True, "lat": True, "lon": True},
    color="country",
    zoom=3.8,
    height=700,
)
fig.update_layout(mapbox_style="open-street-map")
fig.update_layout(legend=dict(title="Country"))
fig.show()
fig.write_image("map_europe_density_filtered.png")


In [30]:
kept_south_europe.columns

Index(['Unnamed: 0', 'souid', 'city', 'country', 'lat', 'lon', 'city_key',
       'years_mode', 'years_with_data', 'nan_fraction',
       'total_rows_post_threshold'],
      dtype='object')

In [31]:
### Step 4: Create south-europe filtered directory from kept_south_europe

SRC_DIR = Path("ECA_nonblend_rr_all_europe_06_10_25")
DST_DIR = Path("ecad_data_south_europe_filtered")
os.makedirs(DST_DIR, exist_ok=True)

copied = []
missing = []

for row in kept_south_europe.itertuples(index=False):
    souid = row.souid
    src = SRC_DIR / f"RR_SOUID{souid}.txt"
    dst = DST_DIR / f"RR_SOUID{souid}.txt"
    if src.exists():
        shutil.copy2(src, dst)
        copied.append(souid)
    else:
        missing.append(souid)

kept_south_europe.to_csv(DST_DIR / "df_candidates_kept.csv", index=False)

print(f"Copied: {len(copied)} files → {DST_DIR}")
if missing:
    print(f"Warning: {len(missing)} souids not found in {SRC_DIR}: {missing}")

Copied: 247 files → ecad_data_south_europe_filtered
